# ML Features: Feature Engineering for Anomaly Detection

**Purpose:** Create engineered features for machine learning model training

**What is ML features?**
Features are inputsto machine learning models. Instead of raw sensor values, we create meaningful metrics that help the model detect anomalies.

**Example:**
Raw data: Temperature = 78°C at 3:00 PM
Features:
- avg_temp_24h = 65°C (24-hour rolling average)
- temp_deviation = 78 - 65 = 13°C (how far from normal)
- hour_of_day = 15 (3 PM = feature for time patterns)
- is_peak_hours = 1 (yes, it's peak hours)

**Why these features?**
- Rolling averages: Smooth out noise, show trends
- Deviations: Show if sensor reading is abnormal
- Temporal features: Equipment behaves differently at different times
- Lag features: Previous readings help predict next reading

**Output:** Feature store table with all engineered features ready for ML model training

In [0]:
# Configuration for Feature Engineering

# INPUT TABLE: Which Silver table has the enriched sensor data?
INPUT_TABLE = "dev.silver.sensor_readings_enriched"

# OUTPUT TABLE: Where we'all save engineered features
OUTPUT_TABLE = "dev.gold.ml_feature_store"

print("=" * 70)
print("CONFIGURATION: Feature Engineering")
print("=" * 70)
print(f"Input table: {INPUT_TABLE}")
print(f"Output tabel: {OUTPUT_TABLE}")

In [0]:
# Load enriched sensor data from Silver layer

enriched_df = spark.read.table(INPUT_TABLE)

print(f"Loaded {enriched_df.count()} records from dev.silver.sensor_readings_enriched silver layer")

print(f"\nColumns available:")
print(enriched_df.columns)

enriched_df.select(
    "equipment_id",
    "sensor_type",
    "timestamp",
    "value",
    "equipment_type",
    "factory_location"
).show(5, truncate=False)

In [0]:
from pyspark.sql.functions import avg, col, round as spark_round
from pyspark.sql.window import Window

# Define a 24-hour rolling window
# This window will partition by equipment_id and sensor_type
# And order by timestamp (to calculate last 24 hours)

window_24h = Window.partitionBy("equipment_id", "sensor_type").orderBy("timestamp")

rolling_features = enriched_df.withColumn(
    "rolling_avg_24h",
    avg(col("value")).over(window_24h)
)

rolling_features.select(
    "equipment_id",
    "sensor_type",
    "timestamp",
    "value",
    "rolling_avg_24h"
).show(10, truncate=False)

In [0]:
from pyspark.sql.functions import lag, col, round as spark_round

# We'all build on rolling_features from Cell 4
# So start with: features_df = rolling_features

window_lag = Window.partitionBy("equipment_id", "sensor_type").orderBy("timestamp")

features_df = rolling_features.withColumn(
    "deviation_from_avg",
    col("value") - col("rolling_avg_24h")
).withColumn(
    "previous_value",
    lag(col("value")).over(window_lag)
).withColumn(
    "value_change",
    col("value") - col("previous_value")
)

# Show sample
features_df.select(
    "equipment_id",
    "sensor_type",
    "timestamp",
    "value",
    "rolling_avg_24h",
    "deviation_from_avg",
    "previous_value",
    "value_change"
).show(10, truncate=False)

In [0]:
from pyspark.sql.functions import hour, dayofweek, col, when

temporal_features = features_df.withColumn(
    "hour_of_day",
    hour(col("timestamp"))
).withColumn(
    "day_of_week",
    dayofweek(col("timestamp"))
).withColumn(
    "is_peak_hour",
    when((hour(col("timestamp")) >= 8 ) & (hour(col("timestamp")) <= 18) , 1).otherwise(0)
).withColumn(
    "is_weekend",
    when((col("day_of_week") == 1) | (col("day_of_week") ==7), 1).otherwise(0)
)

# Show sample
temporal_features.select(
    "timestamp",
    "hour_of_day",
    "day_of_week",
    "is_peak_hour",
    "is_weekend"
).show(10, truncate=False)

In [0]:
# Select final features for ML model

final_features = temporal_features.select(
    # Equipment identification
    "equipment_id",
    "equipment_type",
    "factory_location",
    "sensor_type",

    # Time information
    "timestamp",
    "hour_of_day",
    "day_of_week",
    "is_peak_hour",
    "is_weekend",

    # Raw sensor value
    "value",

    # Engineered features
    "rolling_avg_24h",
    "deviation_from_avg",
    "previous_value",
    "value_change"
).dropna() # Remove rows with NULL values (from lag funciton)

print(f"Final feature store has {final_features.count()} records")
print(f"\nFeature columns:")
print(final_features.columns)

# Show sample
final_features.show(5, truncate=False)

In [0]:
# Write features to Gold layer

print(f"Writing {final_features.count()} feature records to {OUTPUT_TABLE}...")

final_features.write \
    .format("delta") \
    .mode('overwrite') \
    .option("overwriteSchema", "true") \
    .saveAsTable(OUTPUT_TABLE)

print(f"Feature store saved!")
print(f"Table: {OUTPUT_TABLE}")
print(f"Records: {final_features.count()}")

In [0]:
from pyspark.ml.feature import StandardScaler, VectorAssembler
from sklearn.ensemble import IsolationForest   
import pandas as pd
from pyspark.sql.functions import col

# Define which columns to use for ML model training
# These are the ENGINEERED featues (not equipment context)
feature_columns = [
    "rolling_avg_24h",
    "deviation_from_avg",
    "value_change",
    "hour_of_day",
    "day_of_week",
    "is_peak_hour",
    "is_weekend"
]

# Step 1: Combine features inot a vector (ML needs vectors, not individual columns)
print("Converting to Pandas for model training")
features_pandas = final_features.select(
    "equipment_id",
    "sensor_type",
    "timestamp",
    "value",
    *feature_columns
).toPandas()

print(f"Converted {len(features_pandas)} rows to Pandas")
print(f"\nData shape: {features_pandas.shape}")

# Step 2: Prepare features (X) for model
X = features_pandas[feature_columns]

print(f"\nFeatures for training:")
print(X.head())

In [0]:
from sklearn.ensemble import IsolationForest

# Step 3: Train Isolation Forest model 
# Why Isolation Forest? It finds outliers/anomalies in data
# contamination=0.05 means "expect 5% of data to be anomalies"

print("Training Isolation Forest mode...")

model = IsolationForest(contamination=0.05, random_state=42)
model.fit(X)

print(f"Modle trained!")

# Step 4: Make predictions (1 = normal, -1 = anomaly)
predictions = model.predict(X)

anomaly_score = model.score_samples(X)

print(f"Predictions made!")
print(f"Normal radings: {sum(predictions == 1)}")
print(f"Anomalies: {sum(predictions == -1)}")

In [0]:
# step 5: Add predictions and anomaly scores to dataframe

# Create a new column with predictions
features_pandas["prediction"] = predictions #1 = normal, -1 = anomaly

# Create a new column with ranomal score (lower = more anomalous)
features_pandas["anomaly_score"] = anomaly_score

# Convert back to Pyspark dataframe
prediction_df = spark.createDataFrame(features_pandas)

print(f"Predictions added to dataframe")
print(f"\nSample predictions:")
prediction_df.select(
    "equipment_id",
    "sensor_type",
    "timestamp",
    "value",
    "rolling_avg_24h",
    "deviation_from_avg",
    "prediction",
    "anomaly_score"
).show(10, truncate=False)

# Show anomalies only
print(f"\n Detected Anomalies (prediction = -1)")
prediction_df.filter("prediction == -1").select(
    "equipment_id",
    "sensor_type",
    "timestamp",
    "value",
    "rolling_avg_24h",
    "anomaly_score"
).orderBy("anomaly_score").show(10, truncate=False)

In [0]:
# Save predictions to Gold layer

predictions_table = "dev.gold.ml_predictions"

print(f"Saving {prediction_df.count()} predictions to {predictions_table}...")

prediction_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(predictions_table)

print(f"Predictions saved!")
print(f"Table: {predictions_table}")
print(f"Records: {prediction_df.count()}")

In [0]:
print("=" * 90)
print("ML FEATURES & ANOMALY DETECTION - COMPLETE!")
print("=" * 90)

# Count anomalies 
total_records = prediction_df.count()
anomalies = prediction_df.filter("prediction == -1").count()
normal = prediction_df.filter("prediction == 1").count()
anomaly_rate = (anomalies / total_records) * 100

print(f"""
    FEATURE ENGINEERING
    Input: 8,832 sensor readings from Silver layer
    Ouput: 8,432 engineered features (dropped 400 NULLs) 

    Features Created:
    ├── rolling_avg_24h: 24-hours rolling average
    ├── deviation_from_avg: Distance from normal
    ├── previous_value: Lag feature (previous readings)
    ├── value_change: How much value changed
    ├── hour_of_day: time-based feature (0-23)
    ├── day_of_week: Day-based feature (1-7)
    ├── is_peak_hour: Peak hours indicator
    └── is_weekend: Weekend indicator

    MODEL TRAINING
    Algorithm: Isolation Forest
    Training records:8,432
    Contamination: 5% (expect 5% anomalies)

    ANOMLAY DETECTION RESULTS:
    Total records: {total_records},
    Normal readings: {normal} ({100-anomaly_rate:.2f}%)
    Anomalies detected: {anomalies} ({anomaly_rate:.2f}%)

    GOLD LAYER TABLES:
    1. ml_feature_store: 8,432 engineered features
    2. ml_predictions: 8,432 predictions + anomaly scores

    wHAT ANOMALIES MEAN:
    prediction = 1: Normal reading 
    prediction = -1: Anomaly detected!
    anomaly_score: Lower = more anomalous

    USE CASES
    - Real-time alerts: When prediction = -1, alert plant manager
    - Maintenance prediction: Anomalies oftern precede failures
    - Equipment health: High anomaly rate = equipment failing
    - Root cause analysis: What sensor readings caused anomaly?

    COMPLETE PIPLINE
    Bronze (raw) -> Silver (clearn) -> Gold (metrics) -> ML (predictions)

    Total tables created: 11
    Total notebooks created: 13
    Complete end-to-end data platform

    """)

print("=" * 90)
print("=" * 90)